# Working with dates and time

Practice notebook for the datafromzero SQL lesson **Working with dates and time**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Look at the type

In [ ]:
con.sql("""
SELECT order_id, ordered_at, typeof(ordered_at) AS type
FROM orders
ORDER BY order_id
LIMIT 3
""").show()

### Step 2. Take a timestamp apart

In [ ]:
con.sql("""
SELECT order_id, ordered_at, year(ordered_at) AS yr, month(ordered_at) AS mo, day(ordered_at) AS dom,
       dayname(ordered_at) AS weekday, isodow(ordered_at) AS dow
FROM orders
ORDER BY order_id
LIMIT 3
""").show()

### Step 3. Subtract two dates

In [ ]:
con.sql("""
SELECT store_name, opened_on, DATE '2026-06-30' - opened_on AS days_open
FROM stores
""").show()

### Step 4. Add and subtract an interval

In [ ]:
con.sql("""
SELECT COUNT(*) AS last_week_orders
FROM orders
WHERE ordered_at >= DATE '2026-07-01' - INTERVAL 7 DAY
  AND ordered_at < DATE '2026-07-01'
""").show()

### Step 5. Cut a timestamp to the start of its day, week and month

In [ ]:
con.sql("""
SELECT order_id, ordered_at,
       CAST(ordered_at AS DATE) AS day,
       CAST(date_trunc('week', ordered_at) AS DATE) AS week_start,
       CAST(date_trunc('month', ordered_at) AS DATE) AS month_start
FROM orders
ORDER BY order_id
LIMIT 3
""").show()

### Step 6. Group by month

In [ ]:
con.sql("""
SELECT CAST(date_trunc('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
FROM orders
GROUP BY CAST(date_trunc('month', ordered_at) AS DATE)
ORDER BY month
""").show()

### Step 7. Keep the last day

In [ ]:
con.sql("""
SELECT COUNT(*) AS june_between
FROM orders
WHERE ordered_at BETWEEN '2026-06-01' AND '2026-06-30'
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS june
FROM orders
WHERE ordered_at >= '2026-06-01' AND ordered_at < '2026-07-01'
""").show()

### Step 8. Flag an incomplete period

In [ ]:
con.sql("""
SELECT CAST(date_trunc('month', ordered_at) AS DATE) AS month,
       COUNT(*) AS orders,
       MAX(CAST(ordered_at AS DATE)) AS last_order_day,
       last_day(MAX(ordered_at)) AS month_end,
       MAX(CAST(ordered_at AS DATE)) < last_day(MAX(ordered_at)) AS is_partial
FROM orders
WHERE ordered_at < TIMESTAMP '2026-06-13'
GROUP BY CAST(date_trunc('month', ordered_at) AS DATE)
ORDER BY month
""").show()

### Step 9. Verify

## Exercise 1. The busiest weekday

The first lines are given. Add the line that groups by the weekday name and the line that puts the busiest weekday first. The cell gives an error until you do.

In [ ]:
con.sql("""
SELECT dayname(ordered_at) AS weekday, COUNT(*) AS orders
FROM orders
""").show()

## Exercise 2. Orders per month at Harbor

Harbor is `store_id` 3. Before you run anything, write one sentence in the first cell: how many rows you expect, and which months. Then write the query.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague says the quietest weeks of the half-year are the week of June 29 and the week of December 29. Run the query, find the flaw, then write a query that works.

In [ ]:
con.sql("""
SELECT CAST(date_trunc('week', ordered_at) AS DATE) AS week_start, COUNT(*) AS orders
FROM orders
GROUP BY CAST(date_trunc('week', ordered_at) AS DATE)
ORDER BY orders
LIMIT 3
""").show()

In [ ]:
# Your corrected query: